# IE Teachers Knowledge Graph Notebook
This Colab-first notebook walks through parsing IE teacher biographies, extracting entities, normalising labels, and building a NetworkX knowledge graph with friendly QA checkpoints.


In [ ]:
# Install dependencies (Restart & Run-All safe)
!pip install -q -r /content/ie-teachers-kg/requirements.txt


In [ ]:
# Global imports and deterministic setup
import os
import sys
import random
import json
from datetime import datetime
from collections import Counter

import numpy as np

random.seed(42)
np.random.seed(42)

REPO = "/content/ie-teachers-kg"
if REPO not in sys.path:
    sys.path.append(REPO)
SRC_PATH = os.path.join(REPO, "src")
if os.path.isdir(SRC_PATH) and SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

os.makedirs(os.path.join(REPO, "outputs"), exist_ok=True)
print("Repo path:", REPO)
print("Output dir:", os.path.join(REPO, "outputs"))


## Section A — Setup
We install the lightweight NLP stack (transformers + spaCy + NetworkX) and register the repo on `sys.path` so `src/` helpers are importable inside Colab.


## Section B — Load Data


In [ ]:
import pandas as pd

DATA_PATH = os.path.join(REPO, "data", "teachers_db_practice.csv")
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} professor bios from {DATA_PATH}")
df[['alias', 'area', 'position', 'full_info']].head()


## Section C — Sectioning, NER, and rule-based extraction
We clean HTML, segment the bios into logical sections, run two multilingual NER models, merge their outputs, and enrich the detections with regex rules (degrees, years, courses).


In [ ]:
from tqdm.auto import tqdm

from parsing import (
    clean_html,
    split_sections,
    iter_bullets,
    parse_corporate_bullet,
    parse_academic_background_bullet,
    parse_academic_experience_bullet,
)
from ner_hf import load_pipelines, ner_on_line
from fusion import fuse_line
from rules import SECTION_STOP_ORGS, ORG_ALIASES, LOCATION_ALIASES

pipes = load_pipelines()
print("Loaded pipelines:", list(pipes.keys()))


In [ ]:
import re
from typing import Callable, Dict, List, Optional

SECTION_PARSERS: Dict[str, Callable[[str], Optional[dict]]] = {
    "corporate_experience": parse_corporate_bullet,
    "academic_experience": parse_academic_experience_bullet,
    "academic_background": parse_academic_background_bullet,
}
SECTION_ORDER = ["corporate_experience", "academic_experience", "academic_background"]

COURSE_PATTERNS = [
    re.compile(r"Adjunct (?:Level [^ ]+ )?Professor of (?P<course>[^,]+)", re.IGNORECASE),
    re.compile(r"Professor of (?P<course>[^,]+)", re.IGNORECASE),
    re.compile(r"taught (?P<course>.+?) at", re.IGNORECASE),
]


def clean_course_name(text: Optional[str]) -> Optional[str]:
    if not text:
        return None
    cleaned = text.replace("“", """).replace("”", """).replace("’", "'")
    cleaned = re.sub(r"\s+", " ", cleaned).strip(" "'.,;:-")
    if not cleaned:
        return None
    tokens = cleaned.split()
    if not (1 <= len(tokens) <= 8):
        return None
    if len(tokens) == 1 and tokens[0].lower() not in {"economics"}:
        return None
    if not any(ch.isalpha() for ch in cleaned):
        return None
    return cleaned


def course_from_line(line: str, parsed_course: Optional[str]) -> Optional[str]:
    for candidate in [parsed_course]:
        cleaned = clean_course_name(candidate)
        if cleaned:
            return cleaned
    for pattern in COURSE_PATTERNS:
        match = pattern.search(line)
        if match:
            cleaned = clean_course_name(match.group("course"))
            if cleaned:
                return cleaned
    return None


def valid_org(name: Optional[str]) -> bool:
    if not name:
        return False
    stripped = name.strip()
    if not stripped or stripped.lower() in SECTION_STOP_ORGS:
        return False
    return sum(ch.isalnum() for ch in stripped) >= 3


In [ ]:
extraction_records = []
university_names = []
company_names = []
location_names = []
all_courses = []

for idx, row in tqdm(enumerate(df.itertuples(index=False)), total=len(df)):
    html = getattr(row, 'full_info', '') or ''
    sections = split_sections(html)
    if not sections:
        sections = {'intro': clean_html(html)}
    prof_id = getattr(row, 'alias', f'prof_{idx}')
    record = {
        'prof_id': prof_id,
        'area': getattr(row, 'area', None),
        'position': getattr(row, 'position', None),
        'raw_sections': sections,
        'studies': [],
        'work': [],
        'courses': [],
    }

    for section_name in SECTION_ORDER:
        section_text = sections.get(section_name, '')
        if not section_text:
            continue
        for line in iter_bullets(section_text):
            parser = SECTION_PARSERS.get(section_name)
            parsed = parser(line) if parser else None
            ner = ner_on_line(line, pipes)
            candidates = fuse_line(section_name, line, parsed, ner)
            for cand in candidates:
                org_label = cand.org_canon or cand.org_raw
                if not valid_org(org_label):
                    continue
                location_label = cand.location_canon or cand.location_raw
                if location_label:
                    location_names.append(location_label)
                base_entry = {
                    'org_canon': cand.org_canon,
                    'location': cand.location_raw,
                    'location_canon': cand.location_canon,
                    'source_section': section_name,
                    'text_span': line,
                    'meta': cand.meta,
                    'sources': cand.meta.get('sources'),
                }
                if cand.relation == 'worked_at':
                    entry = {
                        **base_entry,
                        'company': org_label,
                        'role': cand.role,
                        'start_year': cand.start_year,
                        'end_year': cand.end_year,
                        'year_bin': cand.year_bin,
                    }
                    record['work'].append(entry)
                    company_names.append(org_label)
                elif cand.relation == 'studied_at':
                    entry = {
                        **base_entry,
                        'university': org_label,
                        'degree': cand.degree_text,
                        'degree_level': cand.degree_level,
                        'field': cand.field,
                        'year': cand.year,
                        'year_bin': cand.year_bin,
                    }
                    record['studies'].append(entry)
                    university_names.append(org_label)
                elif cand.relation == 'teaches':
                    course_name = course_from_line(line, cand.course)
                    if not course_name:
                        continue
                    entry = {
                        **base_entry,
                        'course': course_name,
                        'center': org_label,
                        'start_year': cand.start_year,
                        'end_year': cand.end_year,
                    }
                    record['courses'].append(entry)
                    all_courses.append(course_name)
    extraction_records.append(record)

print(f"Stored {len(extraction_records)} extraction records")


## Section D — Normalisation & Canonical labels
We clean organisation and location names, cluster near-duplicates with RapidFuzz, and map degrees to a controlled vocabulary.


In [ ]:
from normalize import normalize_name, cluster_and_canonicalize

CANON_DEGREES = {
    'PHD': 'PhD',
    'DOCTOR': 'PhD',
    'MSC': 'MSc',
    'MS': 'MSc',
    'MA': 'MA',
    'MBA': 'MBA',
    'BSC': 'BSc',
    'BA': 'BA',
    'MENG': 'MEng',
    'BENG': 'BEng',
}

uni_map = cluster_and_canonicalize(university_names, stopwords=SECTION_STOP_ORGS, alias_map=ORG_ALIASES) if university_names else {}
comp_map = cluster_and_canonicalize(company_names, stopwords=SECTION_STOP_ORGS, alias_map=ORG_ALIASES) if company_names else {}
loc_map = cluster_and_canonicalize(location_names, stopwords=SECTION_STOP_ORGS, alias_map=LOCATION_ALIASES) if location_names else {}
org_map = {**comp_map, **uni_map}

print("University clusters:", json.dumps(uni_map, indent=2)[:500])
print("Company clusters:", json.dumps(comp_map, indent=2)[:500])
print("Location clusters:", json.dumps(loc_map, indent=2)[:500])


def canonical_degree(name):
    if not name:
        return None
    key = name.upper().replace('.', '')
    return CANON_DEGREES.get(key)


def canonical_location(value):
    if not value:
        return None
    norm = normalize_name(value)
    return loc_map.get(norm, norm)


def is_valid_course(name):
    if not name:
        return False
    stripped = name.strip(" "'")
    if len(stripped) < 2:
        return False
    return any(ch.isalnum() for ch in stripped)


for record in extraction_records:
    cleaned_studies = []
    for study in record['studies']:
        uni = study.get('university')
        uni_norm = normalize_name(uni) if uni else None
        study['university_norm'] = uni_norm
        study['university_canon'] = uni_map.get(uni_norm, uni_norm)
        study['location_canon'] = canonical_location(study.get('location')) or study.get('location_canon')
        study['degree_canon'] = study.get('degree_level') or canonical_degree(study.get('degree') or study.get('degree_text'))
        if not valid_org(study.get('university_canon')):
            continue
        cleaned_studies.append(study)
    record['studies'] = cleaned_studies

    cleaned_work = []
    for work in record['work']:
        comp = work.get('company')
        comp_norm = normalize_name(comp) if comp else None
        work['company_norm'] = comp_norm
        work['company_canon'] = comp_map.get(comp_norm, comp_norm)
        work['location_canon'] = canonical_location(work.get('location')) or work.get('location_canon')
        if not valid_org(work.get('company_canon')):
            continue
        cleaned_work.append(work)
    record['work'] = cleaned_work

    cleaned_courses = []
    for course in record['courses']:
        center = course.get('center')
        center_norm = normalize_name(center) if center else None
        course['center_norm'] = center_norm
        course['center_canon'] = org_map.get(center_norm, center_norm)
        course['location_canon'] = canonical_location(course.get('location')) or course.get('location_canon')
        if not is_valid_course(course.get('course')):
            continue
        cleaned_courses.append(course)
    record['courses'] = cleaned_courses


In [ ]:
# Acceptance tests for courses, degrees, and canonicalization
madgar = next(rec for rec in extraction_records if rec['prof_id'] == 'Appius Aemilius Cicero')

ie_course = next(c for c in madgar['courses'] if (c.get('center_canon') or c.get('center')) == 'IE Business School')
assert ie_course['course'] == 'Economics'
assert ie_course.get('location_canon') == 'Spain'
assert ie_course.get('start_year') == 2018
assert (ie_course.get('meta') or {}).get('end_year_text', '').lower() == 'present'

kent_course = next(c for c in madgar['courses'] if 'Kent State University' in (c.get('center_canon') or c.get('center', '')))
assert kent_course['course'] == 'Economics'
assert kent_course.get('location_canon') == 'USA'
assert kent_course.get('start_year') == 2005
assert kent_course.get('end_year') == 2010

ie_mba = next(s for s in madgar['studies'] if (s.get('university_canon') or s.get('university')) == 'IE Business School')
brown_mba = next(s for s in madgar['studies'] if 'Brown University' in (s.get('university_canon') or s.get('university', '')))
assert ie_mba.get('degree_canon') == brown_mba.get('degree_canon') == 'MBA'
assert ie_mba.get('year') == brown_mba.get('year') == 2018
assert ie_mba.get('location_canon') == 'Spain'
assert brown_mba.get('location_canon') == 'USA'

uni_names = {s.get('university_canon') for rec in extraction_records for s in rec['studies'] if s.get('university_canon')}
assert not {u.lower() for u in uni_names if u}.intersection(SECTION_STOP_ORGS)

assert loc_map.get(normalize_name('Spaing')) == 'Spain'
assert uni_map.get(normalize_name('U. de Navarra')) == 'Universidad de Navarra'
assert uni_map.get(normalize_name('University of Navarra')) == 'Universidad de Navarra'
assert uni_map.get(normalize_name('MBA IE')) == 'IE Business School'

print('Acceptance checks passed for courses, degrees, and canonicalization.')


In [ ]:
import os
import networkx as nx

gexf_path = os.path.join(REPO, 'outputs', 'graph.gexf')
assert os.path.exists(gexf_path), 'GEXF file missing'
nx.read_gexf(gexf_path)
print('GEXF file is present and readable:', gexf_path)


## Section E — Graph building & artefacts


In [ ]:
from graph_utils import (
    new_graph,
    add_professor,
    add_university,
    add_company,
    add_course,
    add_degree,
    add_location,
    link_studied_at,
    link_worked_at,
    link_teaches,
    link_located_in,
    save_graph,
    top_k_by_degree,
)
import networkx as nx

G = new_graph()

for record in extraction_records:
    prof_node = add_professor(G, record['prof_id'], area=record.get('area'), position=record.get('position'))
    for study in record['studies']:
        univ = study.get('university_canon') or study.get('university_norm')
        if not univ:
            continue
        univ_node = add_university(G, univ, location=study.get('location_canon') or study.get('location'))
        if study.get('degree_canon'):
            add_degree(G, study['degree_canon'], field=study.get('field'))
        if study.get('location_canon'):
            link_located_in(G, univ_node, study['location_canon'], 'university')
        link_studied_at(
            G,
            prof_node,
            univ_node,
            degree=study.get('degree_canon') or study.get('degree_level') or study.get('degree'),
            field=study.get('field'),
            year=study.get('year'),
            year_bin=study.get('year_bin'),
            source_section=study.get('source_section', 'unknown'),
            text_span=study.get('text_span'),
            meta=study.get('meta'),
        )
    for work in record['work']:
        comp = work.get('company_canon') or work.get('company_norm')
        if not comp:
            continue
        comp_node = add_company(G, comp, location=work.get('location_canon') or work.get('location'))
        if work.get('location_canon'):
            link_located_in(G, comp_node, work['location_canon'], 'company')
        link_worked_at(
            G,
            prof_node,
            comp_node,
            role=work.get('role'),
            start_year=work.get('start_year'),
            end_year=work.get('end_year'),
            year_bin=work.get('year_bin'),
            source_section=work.get('source_section', 'unknown'),
            text_span=work.get('text_span'),
            meta=work.get('meta'),
        )
    for course in record['courses']:
        name = course.get('course')
        if not name:
            continue
        course_node = add_course(G, name)
        link_teaches(
            G,
            prof_node,
            course_node,
            center=course.get('center_canon') or course.get('center'),
            location=course.get('location_canon') or course.get('location'),
            start_year=course.get('start_year'),
            end_year=course.get('end_year'),
            source_section=course.get('source_section', 'unknown'),
            text_span=course.get('text_span'),
            meta=course.get('meta'),
        )

out_dir = os.path.join(REPO, 'outputs')
save_graph(G, out_dir)
print(f"Graph saved to {out_dir}")
print(nx.info(G))
print("Top universities:", top_k_by_degree(G, 'University'))
print("Top companies:", top_k_by_degree(G, 'Company'))


In [ ]:
import random
sampled = random.sample(extraction_records, min(10, len(extraction_records)))
for item in sampled:
    print(json.dumps(item, indent=2)[:1000])
    print('-' * 80)


## Section F — Quick visualisation


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

professors = [n for n, data in G.nodes(data=True) if data.get('type') == 'Professor']
selected = professors[:30]
sub_nodes = set(selected)
for prof in selected:
    sub_nodes.update(G.neighbors(prof))
H = G.subgraph(sub_nodes).copy()
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(H, seed=42)
color_map = []
for node in H:
    node_type = G.nodes[node].get('type')
    color_map.append({
        'Professor': '#1f77b4',
        'University': '#ff7f0e',
        'Company': '#2ca02c',
        'Course': '#d62728',
        'Location': '#9467bd',
        'Degree': '#8c564b',
    }.get(node_type, '#7f7f7f'))
nx.draw(H, pos, with_labels=False, node_color=color_map, node_size=120)
plt.title('Mini knowledge subgraph (first ~30 professors)')
plt.show()


## Section G — Export ZIP deliverable


In [ ]:
import shutil
import tempfile

stamp = datetime.utcnow().strftime('%Y%m%d')
zip_base = os.path.join(REPO, f'ie-teachers-kg_submit_{stamp}')
with tempfile.TemporaryDirectory() as tmp:
    targets = [
        ('notebooks', 'main.ipynb'),
        ('data', 'teachers_db_practice.csv'),
        ('outputs', 'nodes.csv'),
        ('outputs', 'edges.csv'),
        ('outputs', 'graph.gexf'),
        ('', 'requirements.txt'),
        ('', 'README.md'),
    ]
    for folder, fname in targets:
        src = os.path.join(REPO, folder, fname) if folder else os.path.join(REPO, fname)
        if os.path.exists(src):
            dst_dir = os.path.join(tmp, folder) if folder else tmp
            os.makedirs(dst_dir, exist_ok=True)
            shutil.copy2(src, os.path.join(dst_dir, fname))
    shutil.make_archive(zip_base, 'zip', tmp)

print(f"Created archive: {zip_base}.zip")


In [ ]:
try:
    from google.colab import files
    files.download(f"{zip_base}.zip")
except Exception as err:
    print("Download hint: run this cell in Colab to download the ZIP.")
    print(err)


## Section H — Documentation


**Pipeline pseudocode**
```
load CSV → iterate rows
  clean HTML → split sections
  run both NER models → merge spans
  attach regex degrees/courses + nearest locations
  normalise org/location strings via RapidFuzz clusters
  populate NetworkX graph with nodes + edges + provenance
persist nodes/edges/gexf → QA prints → build Colab ZIP deliverable
```


In [ ]:
degree_counter = Counter([
    study.get('degree_canon') or study.get('degree')
    for record in extraction_records for study in record['studies']
    if study.get('degree_canon') or study.get('degree')
])
course_counter = Counter(all_courses)
findings = [
    f"Processed {len(extraction_records)} professors with {len(G.nodes())} nodes and {len(G.edges())} edges in the KG.",
    f"Most common degrees: {degree_counter.most_common(3)}",
    f"Top courses mentioned: {course_counter.most_common(3)}",
    f"Top universities by degree centrality: {top_k_by_degree(G, 'University')[:3]}",
]
for item in findings:
    print(f"- {item}")
